In [1]:
import pandas as pd

df = pd.read_csv('processed/processed_data_with_actr_disp.csv')

In [2]:
df.isna().sum()

asset_id             0
user_index           0
super_asset_nm       0
genre                0
smry              4280
ct_cl                0
view_ratio           0
weekday_sin          0
weekday_cos          0
hour_sin             0
hour_cos             0
u_idx                0
i_idx                0
weight               0
actr_disp            0
dtype: int64

In [3]:
df.dropna(subset = ['smry'], inplace = True)

### ALS

#### ID 매핑

In [4]:
u2i = {u:i for i,u in enumerate(df["user_index"].unique())}
i2i = {a:i for i,a in enumerate(df["asset_id"].unique())}
df["u_idx"] = df["user_index"].map(u2i)
df["i_idx"] = df["asset_id"].map(i2i)

In [5]:
from scipy.sparse import coo_matrix
import pandas as pd
import numpy as np

# view_ratio를 4개 구간으로 나누고(0-0.25, 0.25-0.5, 0.5-0.75, 0.75-1) 가중치 부여
bins = [0, 0.25, 0.5, 0.75, 1.0]
labels = [1.0, 2.0, 3.0, 4.0]  # 각 구간에 해당하는 가중치

# pd.cut을 사용하여 view_ratio를 구간으로 나누고 가중치 할당
df["weight"] = pd.cut(df["view_ratio"], bins=bins, labels=labels, include_lowest=True)
df["weight"] = df["weight"].astype(float)  # 가중치를 float 타입으로 변환

rows = df["i_idx"]          # implicit ALS는 (item × user) 구조 선호
cols = df["u_idx"]
data = df["weight"]         # 구간별 가중치 (1.0, 2.0, 3.0, 4.0)
item_user = coo_matrix((data, (rows, cols)))

In [6]:
from implicit.als import AlternatingLeastSquares
import scipy.sparse as sparse

# ALS 모델 설정
# factors: 잠재 요인(latent factor)의 수
# regularization: 정규화 파라미터
# iterations: 학습 반복 횟수
# calculate_training_loss: 학습 손실 계산 여부
model = AlternatingLeastSquares(factors=100, 
                               regularization=0.1,
                               iterations=50, 
                               calculate_training_loss=True,
                               use_gpu=False)  # GPU가 있으면 True로 설정

# 모델 학습
# item_user 매트릭스를 사용하여 학습
model.fit(item_user, show_progress=True)

# 학습 후 item factors 및 user factors 확인
item_factors = model.item_factors  # 아이템의 잠재 요인 (n_items x factors)
user_factors = model.user_factors  # 사용자의 잠재 요인 (n_users x factors)
print(f"Item factors shape: {item_factors.shape}")
print(f"User factors shape: {user_factors.shape}")


c:\Users\user\anaconda3\Lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: Intel MKL BLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'MKL_NUM_THREADS=1' or by callng 'threadpoolctl.threadpool_limits(1, "blas")'. Having MKL use a threadpool can lead to severe performance issues
  check_blas_config()
c:\Users\user\anaconda3\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 3.5118110179901123 seconds
  warnings.warn(


  0%|          | 0/50 [00:00<?, ?it/s]

Item factors shape: (621933, 100)
User factors shape: (262708, 100)


In [7]:
import pathlib, pickle

# 7-1) 잠재 요인 행렬
np.save("C:/OPS/hybrid_filtering/files/ALS/user_factors.npy", user_factors)
np.save("C:/OPS/hybrid_filtering/files/ALS/item_factors.npy", item_factors)

# 7-2) 전체 객체 + 매핑까지 pickle 직렬화
payload = {
    "model": model,
    "user_map": u2i,
    "item_map": i2i,
    "bucket_bins": bins,
}
with open("C:/OPS/hybrid_filtering/files/ALS/als_full.pkl", "wb") as f:
    pickle.dump(payload, f)

### 콘텐츠 기반 추천 (Content-based Recommendation)

#### 1. 형태소 분석 및 TF-IDF 벡터화 준비

In [8]:
import pandas as pd
import numpy as np
import os
from mecab import MeCab
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle
import time
import json

# 체크포인트 관리를 위한 함수
class CheckpointManager:
    def __init__(self, checkpoint_path='C:/OPS/hybrid_filtering/files/TF-IDF/checkpoint.json'):
        self.checkpoint_path = checkpoint_path
        self.checkpoint = self._load()
    def _load(self):
        if os.path.exists(self.checkpoint_path):
            with open(self.checkpoint_path, 'r', encoding='utf-8') as f:
                return json.load(f)
        return {"stage": "start", "processed_items": []}
    def save(self, stage=None, processed_items=None):
        if stage:
            self.checkpoint["stage"] = stage
        if processed_items is not None:
            self.checkpoint["processed_items"] = processed_items
        with open(self.checkpoint_path, 'w', encoding='utf-8') as f:
            json.dump(self.checkpoint, f, ensure_ascii=False, indent=2)
    @property
    def processed(self):
        return set(self.checkpoint.get("C:/OPS/hybrid_filtering/files/TF-IDF/processed_items", []))
    @property
    def stage(self):
        return self.checkpoint.get("stage", "start")

ckpt = CheckpointManager()
print(f"✅ 현재 stage: {ckpt.stage} | 이미 처리된 group_id 수: {len(ckpt.processed)}")

# 콘텐츠 데이터 중복 확인
content_df = df[["asset_id", "super_asset_nm", "smry", "genre", "actr_disp"]].drop_duplicates()
print(f"고유 콘텐츠 아이템 수: {content_df.shape[0]}")

# 불용어 목록 정의
stopwords = [
    "이", "그", "저", "것", "수", "등", "를", "을", "에", "에서", "하다", "있다", "되다", 
    "가다", "오다", "이다", "그렇다", "그것", "무엇", "어떤", "이런", "그런", "저런", "때", 
    "데", "곳", "그리고", "그래서", "그러나", "하지만", "또는", "또한", "및", "한", "두", 
    "세", "네", "이런", "저런", "그런", "어떤", "무슨", "어느", "이러한", "저러한", "그러한",
    "어떠한", "이와", "그와", "저와", "이에", "그에", "저에", "이가", "그가", "저가", "이의", 
    "그의", "저의", "의", "에는", "에서는", "로", "로써", "보다", "없다", "않다", "말다", 
    "통해", "위해", "때문", "인해", "위하여", "대하여", "관하여", "대한", "관한"
]

✅ 현재 stage: start | 이미 처리된 group_id 수: 0
고유 콘텐츠 아이템 수: 262708


#### 2. MeCab 형태소 분석 함수 정의

In [9]:
mecab = MeCab()

def analyze_text(text):
    if pd.isna(text) or text == '':
        return ''
    try:
        pos_tags = mecab.pos(text)
        words = [w for w, t in pos_tags
                 if (t.startswith(('NN', 'VA', 'VV'))
                     and len(w) > 1 and w not in stopwords)]
        return ' '.join(words)
    except Exception as e:
        print(f"[⚠️] 형태소 분석 오류: {e} → {text[:30]}")
        return ''

def process_row(row):
    genre_twice = (row['genre'] + ' ' + row['genre']) if pd.notna(row['genre']) else ''
    texts = [row['smry'] or '', genre_twice, row['actr_disp'] or '']
    return ' '.join(filter(None, map(analyze_text, texts)))

# -------------------- 4. 그룹화 & 대표 아이템 추출 --------------------
def group_content(df_):
    grp = df_.groupby(['super_asset_nm', 'smry'])['asset_id']\
             .apply(list).reset_index()
    grp['group_id'] = range(len(grp))
    id2grp = {aid: g for _, (ids, g) in grp[['asset_id','group_id']].iterrows()
              for aid in ids}
    df_ = df_.copy()
    df_['group_id'] = df_['asset_id'].map(id2grp)
    return df_, grp

content_df, _ = group_content(content_df)
rep_df = content_df.drop_duplicates('group_id')
rep_map = rep_df.set_index('group_id').to_dict('index')   # ← 추가
total_items = len(rep_map)                                # 총 개수도 여기서 계산

#### 3. 콘텐츠 그룹화 및 형태소 분석 수행

In [10]:
# -------------------- 5. 이미 분석된 텍스트 불러오기 --------------------
ckpt = CheckpointManager()   # 새 인스턴스 → 메모리 내용 초기화

analyzed_texts = {}
if os.path.exists('C:/OPS/hybrid_filtering/files/TF-IDF/analyzed_texts.pkl'):
    with open('C:/OPS/hybrid_filtering/files/TF-IDF/analyzed_texts.pkl', 'rb') as f:
        analyzed_texts = pickle.load(f)
print(f"🔄 이전까지 저장된 analyzed_texts: {len(analyzed_texts)}개")

# -------------------- 6. 남은 작업 계산 --------------------
processed_ids = set(int(x) for x in ckpt.processed)
remaining_ids = [gid for gid in rep_map.keys() if gid not in processed_ids]
print(f"🚀 이번에 새로 처리할 그룹: {len(remaining_ids)} / {total_items}")

# -------------------- 7. 형태소 분석 루프 --------------------
start = time.time()
batch_counter = 0
for gid in remaining_ids:
    row_info = rep_map.get(gid)          # dict 조회 → 없으면 None
    if row_info is None:                 # rep_df에 존재하지 않는 "유령 id"
        print(f"⚠️ group_id {gid} 누락 → 건너뜀")
        continue

    row = pd.Series(row_info)            # dict → Series (process_row가 Series 예상)
    analyzed_texts[gid] = process_row(row)
    processed_ids.add(gid); batch_counter += 1

    if batch_counter % 100 == 0:
        with open('analyzed_texts.pkl', 'wb') as f:
            pickle.dump(analyzed_texts, f)
        ckpt.save(stage="processing", processed_items=list(processed_ids))
        print(f"💾 {len(processed_ids):,}/{total_items:,} 완료 "
              f"({time.time()-start:.1f}s 경과)")

# -------------------- 8. 최종 저장 --------------------
with open('C:/OPS/hybrid_filtering/files/TF-IDF/analyzed_texts.pkl', 'wb') as f:
    pickle.dump(analyzed_texts, f)

final_stage = "finished" if len(processed_ids) == total_items else "processing"
ckpt.save(stage=final_stage, processed_items=list(processed_ids))
print(f"🎉 형태소 분석 완료: {len(processed_ids):,}/{total_items:,} (stage={final_stage})")

🔄 이전까지 저장된 analyzed_texts: 0개
🚀 이번에 새로 처리할 그룹: 194379 / 194379
💾 100/194,379 완료 (1.1s 경과)
💾 200/194,379 완료 (1.5s 경과)
💾 300/194,379 완료 (1.9s 경과)
💾 400/194,379 완료 (2.2s 경과)
💾 500/194,379 완료 (2.6s 경과)
💾 600/194,379 완료 (2.8s 경과)
💾 700/194,379 완료 (3.1s 경과)
💾 800/194,379 완료 (3.6s 경과)
💾 900/194,379 완료 (4.0s 경과)
💾 1,000/194,379 완료 (4.4s 경과)
💾 1,100/194,379 완료 (5.0s 경과)
💾 1,200/194,379 완료 (5.3s 경과)
💾 1,300/194,379 완료 (5.4s 경과)
💾 1,400/194,379 완료 (5.5s 경과)
💾 1,500/194,379 완료 (5.6s 경과)
💾 1,600/194,379 완료 (5.8s 경과)
💾 1,700/194,379 완료 (6.0s 경과)
💾 1,800/194,379 완료 (6.1s 경과)
💾 1,900/194,379 완료 (6.2s 경과)
💾 2,000/194,379 완료 (6.4s 경과)
💾 2,100/194,379 완료 (6.5s 경과)
💾 2,200/194,379 완료 (6.7s 경과)
💾 2,300/194,379 완료 (6.8s 경과)
💾 2,400/194,379 완료 (7.0s 경과)
💾 2,500/194,379 완료 (7.1s 경과)
💾 2,600/194,379 완료 (7.3s 경과)
💾 2,700/194,379 완료 (7.4s 경과)
💾 2,800/194,379 완료 (7.5s 경과)
💾 2,900/194,379 완료 (7.6s 경과)
💾 3,000/194,379 완료 (7.8s 경과)
💾 3,100/194,379 완료 (7.9s 경과)
💾 3,200/194,379 완료 (8.1s 경과)
💾 3,300/194,379 완료 (8.4s 경과

#### 4. TF-IDF 벡터화

In [11]:
# ───────────────── TF-IDF 벡터화 전용 함수 ─────────────────
def run_tfidf_vectorization(ckpt,
                             analyzed_path="analyzed_texts.pkl",
                             model_dir="C:/OPS/hybrid_filtering/files/TF-IDF/"):
    """
    형태소 분석 결과(analyzed_texts.pkl)를 TF-IDF 벡터로 변환하고
    벡터라이저·문서벡터를 models/ 아래 저장한다.
    """
    # 0) 이전에 완료했으면 스킵
    if ckpt.stage == "vectorized":
        print("ℹ️ 이미 TF-IDF 벡터화가 완료된 상태입니다.")
        return

    # 1) 폴더 준비 --------------------------------------------------
    os.makedirs(model_dir, exist_ok=True)

    # 2) 형태소 분석 결과 로드 --------------------------------------
    if not os.path.exists(analyzed_path):
        raise FileNotFoundError("❌ analyzed_texts.pkl 이 없습니다. "
                                "먼저 형태소 분석 단계를 수행해 주세요.")

    with open(analyzed_path, "rb") as f:
        analyzed_texts = pickle.load(f)
    print(f"✅ 분석 텍스트 로드: {len(analyzed_texts)}개")

    # 3) 데이터 분리 ------------------------------------------------
    group_ids = list(analyzed_texts.keys())
    documents = [analyzed_texts[gid] for gid in group_ids]

    # 4) TF-IDF 벡터라이저 학습 ------------------------------------
    tfidf = TfidfVectorizer(
        min_df=3,
        max_df=0.9,
        max_features=5000,
        ngram_range=(1, 2)
    )
    tfidf_matrix = tfidf.fit_transform(documents)
    print(f"✅ TF-IDF 행렬 크기: {tfidf_matrix.shape}")

    # 5) 모델·벡터 저장 -------------------------------------------
    vec_path   = os.path.join(model_dir, "C:/OPS/hybrid_filtering/files/TF-IDF/tfidf_vectorizer.pkl")
    gv_path    = os.path.join(model_dir, "C:/OPS/hybrid_filtering/files/TF-IDF/group_vectors.pkl")

    with open(vec_path, "wb") as f:
        pickle.dump(tfidf, f)

    group_vectors = {gid: tfidf_matrix[i] for i, gid in enumerate(group_ids)}
    with open(gv_path, "wb") as f:
        pickle.dump(group_vectors, f)

    print(f"💾 저장 완료 → {vec_path}, {gv_path}")

    # 6) 체크포인트 갱신 ------------------------------------------
    ckpt.save(stage="vectorized")
    print("🎉 TF-IDF 벡터화 단계 완료 & checkpoint 갱신")

In [12]:
run_tfidf_vectorization(ckpt)

✅ 분석 텍스트 로드: 194300개
✅ TF-IDF 행렬 크기: (194300, 5000)
💾 저장 완료 → C:/OPS/hybrid_filtering/files/TF-IDF/tfidf_vectorizer.pkl, C:/OPS/hybrid_filtering/files/TF-IDF/group_vectors.pkl
🎉 TF-IDF 벡터화 단계 완료 & checkpoint 갱신


## 하이브리드 추천

In [13]:
# asset_id와 group_id 매핑 생성
asset_to_group = dict(content_df[['asset_id', 'group_id']].values)

# df에 group_id 추가
df['group_id'] = df['asset_id'].map(asset_to_group)

# 결측치 확인 및 처리 (선택 사항)
if df['group_id'].isna().any():
    print(f"경고: {df['group_id'].isna().sum()}개의 asset_id가 content_df에 없어 group_id가 누락됨")

### 1. 데이터 로드

In [14]:
import pandas as pd
import numpy as np
import pickle
from scipy import sparse
from implicit.als import AlternatingLeastSquares
from sklearn.metrics.pairwise import cosine_similarity
import os

# ALS 모델 로드
with open("C:/OPS/hybrid_filtering/files/ALS/als_full.pkl", "rb") as f:
    als_payload = pickle.load(f)
als_model = als_payload["model"]
user_map = als_payload["user_map"]  # user_index -> u_idx
item_map = als_payload["item_map"]  # asset_id -> i_idx

# 인덱스 역매핑 생성
index_to_item = {i: a for a, i in item_map.items()}

# TF-IDF 데이터 로드
with open("C:/OPS/hybrid_filtering/files/TF-IDF/group_vectors.pkl", "rb") as f:
    group_vectors = pickle.load(f)
group_ids = list(group_vectors.keys())
tfidf_matrix = sparse.vstack([group_vectors[gid] for gid in group_ids])

# 매핑 준비
content_df = df[["asset_id", "super_asset_nm", "smry", "genre", "actr_disp"]].drop_duplicates()
content_df, _ = group_content(content_df)  # group_content 함수는 제공된 코드에서 가져옴
asset_to_group = content_df.set_index('asset_id')['group_id'].to_dict()
asset_to_super = content_df.set_index('asset_id')['super_asset_nm'].to_dict()
rep_df = content_df.drop_duplicates('group_id')
rep_map = rep_df.set_index('group_id').to_dict('index')

# 사용자 시청 이력 및 그룹 매핑
user_watched = df.groupby('user_index')['asset_id'].apply(list).to_dict()
user_groups = df.groupby('user_index')['group_id'].apply(lambda x: list(set(x))).to_dict()

In [15]:
# ALS 모델 로드
with open("C:/OPS/hybrid_filtering/files/ALS/als_full.pkl", "rb") as f:
    als_payload = pickle.load(f)
als_model = als_payload["model"]
user_map = als_payload["user_map"]  # user_index -> u_idx
item_map = als_payload["item_map"]  # asset_id -> i_idx

# 인덱스 역매핑 생성
index_to_item = {i: a for a, i in item_map.items()}

# TF-IDF 데이터 로드
with open("C:/OPS/hybrid_filtering/files/TF-IDF/group_vectors.pkl", "rb") as f:
    group_vectors = pickle.load(f)
group_ids = list(group_vectors.keys())
tfidf_matrix = sparse.vstack([group_vectors[gid] for gid in group_ids])

# 매핑 준비
content_df = df[["asset_id", "super_asset_nm", "smry", "genre", "actr_disp"]].drop_duplicates()
content_df, _ = group_content(content_df)  # group_content 함수는 제공된 코드에서 가져옴
asset_to_group = content_df.set_index('asset_id')['group_id'].to_dict()
asset_to_super = content_df.set_index('asset_id')['super_asset_nm'].to_dict()
rep_df = content_df.drop_duplicates('group_id')
rep_map = rep_df.set_index('group_id').to_dict('index')

# 사용자 시청 이력 및 그룹 매핑
user_watched = df.groupby('user_index')['asset_id'].apply(list).to_dict()
user_groups = df.groupby('user_index')['group_id'].apply(lambda x: list(set(x))).to_dict()

# user_map과 item_map 크기 확인
max_users = max(user_map.values()) + 1
max_items = max(item_map.values()) + 1
print(f"최대 사용자 수 (user_map): {max_users}")
print(f"최대 아이템 수 (item_map): {max_items}")

# ALS용 user_items 행렬 생성
user_items = sparse.coo_matrix(
    (df["weight"], (df["u_idx"], df["i_idx"])),
    shape=(max_users, max_items)  # 명시적으로 크기 지정
)
user_items = user_items.tocsr()  # COO를 CSR로 변환
print(f"수정된 user_items shape: {user_items.shape}")

# shape 일치 여부 확인
if user_items.shape[0] != max_users:
    raise ValueError(f"user_items 행 수({user_items.shape[0]})가 user_map의 최대 사용자 수({max_users})와 다릅니다!")
if user_items.shape[1] != max_items:
    raise ValueError(f"user_items 열 수({user_items.shape[1]})가 item_map의 최대 아이템 수({max_items})와 다릅니다!")

최대 사용자 수 (user_map): 621933
최대 아이템 수 (item_map): 262708
수정된 user_items shape: (621933, 262708)


### 2. ALS 추천 함수

In [16]:
def get_als_recommendations(user_index, N=100):
    """ALS 모델로 상위 N개 추천을 반환"""
    if user_index not in user_map:
        raise ValueError(f"user_index {user_index}가 user_map에 없습니다!")
    u_idx = user_map[user_index]
    print(f"사용자 user_index: {user_index}, u_idx: {u_idx}")
    itemids, scores = als_model.recommend(
        u_idx, 
        user_items, 
        N=N, 
        filter_already_liked_items=True
    )
    rec_asset_ids = [index_to_item[itemid] for itemid in itemids]
    return rec_asset_ids, scores

In [17]:
def get_als_recommendations_1(user_index, N=100):
    """ALS 모델로 상위 N개 추천을 반환"""
    if user_index not in user_map:
        raise ValueError(f"user_index {user_index}가 user_map에 없습니다!")
    u_idx = user_map[user_index]
    print(f"사용자 user_index: {user_index}, u_idx: {u_idx}")
    
    # 단일 사용자의 user_items 행 추출
    user_items_single = user_items[u_idx:u_idx+1]  # u_idx에 해당하는 단일 행 추출
    print(f"단일 사용자 user_items shape: {user_items_single.shape}")
    
    itemids, scores = als_model.recommend(
        u_idx, 
        user_items_single,  # 단일 사용자 행렬 전달
        N=N, 
        filter_already_liked_items=True
    )
    rec_asset_ids = [index_to_item[itemid] for itemid in itemids]
    return rec_asset_ids, scores

### 3. 하이브리드 추천 함수 (ALS + Content 기반 결합)

In [18]:
def hybrid_recommend(user_index, N=100, M=100, K=10, alpha=0.5):
    """
    하이브리드 추천 함수
    """
    watched = set(user_watched.get(user_index, []))
    user_group_ids = user_groups.get(user_index, [])
    
    # ALS 추천
    als_rec_asset_ids, als_scores = get_als_recommendations(user_index, N=N)
    
    # 콘텐츠 기반 추천
    if user_group_ids:
        user_group_vectors = sparse.vstack([group_vectors[gid] for gid in user_group_ids])
        similarity_matrix = cosine_similarity(tfidf_matrix, user_group_vectors)
        similarity_scores = np.max(similarity_matrix, axis=1)
        sorted_indices = np.argsort(similarity_scores)[::-1]
        filtered_indices = [idx for idx in sorted_indices if group_ids[idx] not in user_group_ids]
        top_m_indices = filtered_indices[:M]
        top_m_group_ids = [group_ids[idx] for idx in top_m_indices]
        content_rec_asset_ids = [rep_map[gid]['asset_id'] for gid in top_m_group_ids]
        content_scores = [similarity_scores[idx] for idx in top_m_indices]
    else:
        content_rec_asset_ids = []
        content_scores = []
    
    # 후보 결합 및 시청 이력 제외
    candidate_asset_ids = list(set(als_rec_asset_ids + content_rec_asset_ids) - watched)
    if not candidate_asset_ids:
        return []
    
    # ALS 점수 계산
    u_idx = user_map[user_index]
    user_factor = als_model.user_factors[u_idx]
    candidate_item_indices = [item_map[aid] for aid in candidate_asset_ids]
    candidate_item_factors = als_model.item_factors[candidate_item_indices]
    als_scores = (user_factor @ candidate_item_factors.T).flatten()
    
    # ALS 점수 정규화
    min_als = np.min(als_scores)
    max_als = np.max(als_scores)
    if max_als > min_als:
        normalized_als = (als_scores - min_als) / (max_als - min_als)
    else:
        normalized_als = np.zeros_like(als_scores)
    
    # 콘텐츠 기반 점수 계산
    if user_group_ids:
        group_to_content_score = dict(zip(group_ids, similarity_scores))
        content_scores = np.array([group_to_content_score[asset_to_group[aid]] 
                                 for aid in candidate_asset_ids])
    else:
        content_scores = np.zeros(len(candidate_asset_ids))
    
    # 하이브리드 점수 계산
    hybrid_scores = alpha * normalized_als + (1 - alpha) * content_scores
    
    # 상위 K개 선택 (super_asset_nm 중복 제외)
    sorted_indices = np.argsort(hybrid_scores)[::-1]
    seen_super = set()
    recommended = []
    for idx in sorted_indices:
        aid = candidate_asset_ids[idx]
        super_nm = asset_to_super[aid]
        if super_nm not in seen_super:
            recommended.append((aid, super_nm, hybrid_scores[idx]))
            seen_super.add(super_nm)
            if len(recommended) == K:
                break
    return recommended

In [19]:
def hybrid_recommend_1(user_index, N=100, M=100, K=10, alpha=0.5):
    """
    하이브리드 추천 함수
    """
    watched = set(user_watched.get(user_index, []))
    user_group_ids = user_groups.get(user_index, [])
    
    # ALS 추천
    als_rec_asset_ids, als_scores = get_als_recommendations_1(user_index, N=N)
    
    # 콘텐츠 기반 추천
    if user_group_ids:
        user_group_vectors = sparse.vstack([group_vectors[gid] for gid in user_group_ids])
        similarity_matrix = cosine_similarity(tfidf_matrix, user_group_vectors)
        similarity_scores = np.max(similarity_matrix, axis=1)
        sorted_indices = np.argsort(similarity_scores)[::-1]
        filtered_indices = [idx for idx in sorted_indices if group_ids[idx] not in user_group_ids]
        top_m_indices = filtered_indices[:M]
        top_m_group_ids = [group_ids[idx] for idx in top_m_indices]
        content_rec_asset_ids = [rep_map[gid]['asset_id'] for gid in top_m_group_ids]
        content_scores = [similarity_scores[idx] for idx in top_m_indices]
    else:
        content_rec_asset_ids = []
        content_scores = []
    
    # 후보 결합 및 시청 이력 제외
    candidate_asset_ids = list(set(als_rec_asset_ids + content_rec_asset_ids) - watched)
    if not candidate_asset_ids:
        return []
    
    # ALS 점수 계산
    u_idx = user_map[user_index]
    user_factor = als_model.user_factors[u_idx]
    candidate_item_indices = [item_map[aid] for aid in candidate_asset_ids]
    candidate_item_factors = als_model.item_factors[candidate_item_indices]
    als_scores = (user_factor @ candidate_item_factors.T).flatten()
    
    # ALS 점수 정규화
    min_als = np.min(als_scores)
    max_als = np.max(als_scores)
    if max_als > min_als:
        normalized_als = (als_scores - min_als) / (max_als - min_als)
    else:
        normalized_als = np.zeros_like(als_scores)
    
    # 콘텐츠 기반 점수 계산
    if user_group_ids:
        group_to_content_score = dict(zip(group_ids, similarity_scores))
        content_scores = np.array([group_to_content_score[asset_to_group[aid]] 
                                 for aid in candidate_asset_ids])
    else:
        content_scores = np.zeros(len(candidate_asset_ids))
    
    # 하이브리드 점수 계산
    hybrid_scores = alpha * normalized_als + (1 - alpha) * content_scores
    
    # 상위 K개 선택 (super_asset_nm 중복 제외)
    sorted_indices = np.argsort(hybrid_scores)[::-1]
    seen_super = set()
    recommended = []
    for idx in sorted_indices:
        aid = candidate_asset_ids[idx]
        super_nm = asset_to_super[aid]
        if super_nm not in seen_super:
            recommended.append((aid, super_nm, hybrid_scores[idx]))
            seen_super.add(super_nm)
            if len(recommended) == K:
                break
    return recommended

### 4. 사용 예시

In [20]:
# 예시 사용자
user_index = df['user_index'].iloc[3]
print(f"선택한 user_index: {user_index}, 매핑된 u_idx: {user_map.get(user_index, '없음')}")
recommendations = hybrid_recommend(user_index, N=100, M=100, K=10, alpha=0.5)

# 결과 출력
print(f"사용자 {user_index}에 대한 추천:")
for aid, super_nm, score in recommendations:
    print(f"asset_id: {aid}, super_asset_nm: {super_nm}, score: {score:.4f}")

선택한 user_index: 3, 매핑된 u_idx: 3
사용자 user_index: 3, u_idx: 3


ValueError: user_items must contain 1 row for every user in userids

In [21]:
# 예시 사용자
user_index = df['user_index'].iloc[3]
print(f"선택한 user_index: {user_index}, 매핑된 u_idx: {user_map.get(user_index, '없음')}")
recommendations = hybrid_recommend_1(user_index, N=100, M=100, K=10, alpha=0.5)

# 결과 출력
print(f"사용자 {user_index}에 대한 추천:")
for aid, super_nm, score in recommendations:
    print(f"asset_id: {aid}, super_asset_nm: {super_nm}, score: {score:.4f}")

선택한 user_index: 3, 매핑된 u_idx: 3
사용자 user_index: 3, u_idx: 3
단일 사용자 user_items shape: (1, 262708)


KeyError: 326999